**DATASET**

In [6]:
from google.colab import files

archivo = files.upload()





Saving animal_data_dirty1.csv to animal_data_dirty1.csv


In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("animal_data_dirty1.csv", sep=";")

display(df.head())

,Animal type,Country,Weight kg,Body Length cm,Gender,Animal code,Latitude,Longitude,Animal name,Observation date,Data compiled by
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03.01.2024,James Johnson
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03.02.2024,James Johnson
2,European bison,Poland,930.0,335.0,male,NaN,52.828845,23.820144,Szefu,01.03.2024,Anne Anthony
3,European bison,Poland,909.0,311.0,not determined,NaN,52.830509,23.826849,NaN,01.03.2024,Anne Anthony
4,European bison™,Poland,581.0,277.0,female,NaN,52.834109,23.807093,NaN,01.03.2024,Anne Anthony


**1. DIAGNOSTICO INICIAL**

In [11]:
# cuántas filas tiene
df.shape

(1011, 11)

In [12]:
#Cuáles son las columnas y qué tipo de dato tiene cada una?
df.dtypes

,0
Animal type,object
Country,object
Weight kg,float64
Body Length cm,float64
Gender,object
Animal code,float64
Latitude,float64
Longitude,float64
Animal name,object
Observation date,object


In [13]:
#Cuántos valores faltantes existen por columna?
df.isnull().sum()

,0
Animal type,20
Country,12
Weight kg,27
Body Length cm,27
Gender,19
Animal code,1011
Latitude,98
Longitude,98
Animal name,959
Observation date,0


In [14]:
#Cuántos registros duplicados existen?
df.duplicated().sum()

np.int64(167)

In [17]:
#Utiliza unique() o value_counts() para identificar categorías inconsistentes o valores sospechosos.
df["Animal type"].value_counts(dropna=False)


,count
Animal type,
red squirrel,543
hedgehog,274
lynx,61
European bison,49
red squirrell,22
NaN,20
red squirel,14
lynx?,10
European bison™,6


In [18]:
df["Country"].value_counts(dropna=False)

,count
Country,
Poland,291
Germany,176
Slovakia,151
Hungary,146
Czech Republic,103
Austria,74
PL,15
NaN,12
HU,11


In [19]:
df["Gender"].value_counts(dropna=False)

,count
Gender,
male,499
female,488
NaN,19
not determined,5


In [20]:
df.describe()

,Weight kg,Body Length cm,Animal code,Latitude,Longitude
count,984.000000,984.000000,0.0,913.000000,913.000000
mean,39.745503,39.107724,NaN,49.393369,18.203280
std,156.290076,58.628601,NaN,7.168900,3.899601
min,-0.252000,-19.000000,NaN,-78.582973,11.074008
25%,0.293000,19.000000,NaN,48.186913,14.384559
50%,0.331500,21.000000,NaN,49.560723,18.944015
75%,0.800000,23.000000,NaN,52.212433,21.033243
max,1100.000000,350.000000,NaN,52.853843,34.896734


Después de realizar el diagnóstico inicial de la base de datos, se identificaron los siguientes problemas:

**Valores faltantes**: se encontraron datos ausentes en diferentes columnas. El caso más significativo es Animal code, que presenta 1.011 valores faltantes, y Animal name, con 959. Esto puede limitar algunos análisis y requiere evaluar si estas variables son necesarias.

**Registros duplicados:** se identificaron 167 registros duplicados, lo que puede generar inconsistencias en el conteo de observaciones de animales y afectar los resultados estadísticos.

**Inconsistencias en las categorías:** algunas especies presentan diferentes formas de escritura, como red squirrel, red squirrell y red squirel. También se encontraron diferencias en los nombres de los países, como Poland y PL, o Hungary y HU.

**Valores numéricos incorrectos:** se encontraron pesos y longitudes negativos, valores que no tienen sentido para las mediciones físicas de los animales.

**Errores geográficos**: algunas coordenadas de latitud presentan valores que no corresponden a la región geográfica del estudio, por lo que deben revisarse antes de utilizarlas en un análisis de biodiversidad.

**2. LIMPIEZA DE DATOS**

In [28]:
df_limpio = df.copy()

In [29]:
for columna in df_limpio.select_dtypes(include="object").columns:
    df_limpio[columna] = df_limpio[columna].str.strip()




In [30]:
#eliminar registros duplicados
print("Duplicados antes:", df_limpio.duplicated().sum())

df_limpio = df_limpio.drop_duplicates()

print("Duplicados después:", df_limpio.duplicated().sum())
print("Dimensiones actuales:", df_limpio.shape)

Duplicados antes: 167
Duplicados después: 0
Dimensiones actuales: (844, 11)


In [32]:
#correccion animales
correcciones_animales = {
    "red squirrell": "red squirrel",
    "red squirel": "red squirrel",
    "lynx?": "lynx",
    "European bison™": "European bison",
    "European bisson": "European bison",
    "European buster": "European bison",
    "ledgehod": "hedgehog",
    "wedgehod": "hedgehog"
}

df_limpio["Animal type"] = df_limpio["Animal type"].replace(correcciones_animales)

In [33]:
print(df_limpio["Animal type"].value_counts(dropna=False))

Animal type
red squirrel      413
hedgehog          278
lynx               71
European bison     63
NaN                19
Name: count, dtype: int64


In [35]:
#correccion paises
correcciones_paises = {
    "PL": "Poland",
    "HU": "Hungary",
    "Hungry": "Hungary",
    "CZ": "Czech Republic",
    "Czech": "Czech Republic",
    "DE": "Germany",
    "IT": "Italy"
}

df_limpio["Country"] = df_limpio["Country"].replace(correcciones_paises)


In [36]:
print(df_limpio["Country"].value_counts(dropna=False))

Country
Poland            192
Germany           178
Slovakia          151
Hungary           118
Czech Republic    112
Austria            74
NaN                11
Australia           4
CC                  4
Name: count, dtype: int64


In [37]:
df_limpio = df_limpio.drop(columns=["Animal code", "Animal name"])

print(df_limpio.columns)

Index(['Animal type', 'Country', 'Weight kg', 'Body Length cm', 'Gender',
       'Latitude', 'Longitude', 'Observation date', 'Data compiled by'],
      dtype='object')


In [39]:
df_limpio["Animal type"] = df_limpio["Animal type"].fillna("Unknown")

df_limpio["Country"] = df_limpio["Country"].fillna("Unknown")

df_limpio["Gender"] = df_limpio["Gender"].fillna("Not determined")

In [40]:
df_limpio.loc[df_limpio["Weight kg"] < 0, "Weight kg"] = np.nan

df_limpio.loc[df_limpio["Body Length cm"] < 0, "Body Length cm"] = np.nan

In [41]:
for columna in ["Weight kg", "Body Length cm"]:
    df_limpio[columna] = df_limpio.groupby("Animal type")[columna].transform(
        lambda x: x.fillna(x.median())
    )

    df_limpio[columna] = df_limpio[columna].fillna(df_limpio[columna].median())

In [42]:
print("Latitudes inválidas:")
print(df_limpio[~df_limpio["Latitude"].between(-90, 90)]["Latitude"])

print("Longitudes inválidas:")
print(df_limpio[~df_limpio["Longitude"].between(-180, 180)]["Longitude"])

Latitudes inválidas:
0     NaN
1     NaN
13    NaN
21    NaN
22    NaN
       ..
956   NaN
963   NaN
968   NaN
969   NaN
970   NaN
Name: Latitude, Length: 92, dtype: float64
Longitudes inválidas:
0     NaN
1     NaN
13    NaN
21    NaN
22    NaN
       ..
956   NaN
963   NaN
968   NaN
969   NaN
970   NaN
Name: Longitude, Length: 92, dtype: float64


In [43]:
df_limpio.loc[
    ~df_limpio["Latitude"].between(-90, 90), "Latitude"
] = np.nan

df_limpio.loc[
    ~df_limpio["Longitude"].between(-180, 180), "Longitude"
] = np.nan

In [44]:
df_limpio["Observation date"] = pd.to_datetime(
    df_limpio["Observation date"],
    errors="coerce"
)

In [45]:
print("Dimensiones finales:", df_limpio.shape)

print("\nValores faltantes:")
print(df_limpio.isnull().sum())

print("\nDuplicados restantes:")
print(df_limpio.duplicated().sum())

print("\nTipos de datos:")
print(df_limpio.dtypes)

Dimensiones finales: (844, 9)

Valores faltantes:
Animal type           0
Country               0
Weight kg             0
Body Length cm        0
Gender                0
Latitude             92
Longitude            92
Observation date    391
Data compiled by      0
dtype: int64

Duplicados restantes:
0

Tipos de datos:
Animal type                 object
Country                     object
Weight kg                  float64
Body Length cm             float64
Gender                      object
Latitude                   float64
Longitude                  float64
Observation date    datetime64[ns]
Data compiled by            object
dtype: object


In [46]:
fechas_invalidas = df_limpio[
    df_limpio["Observation date"].isna()
]

print("Cantidad de fechas inválidas:", len(fechas_invalidas))

Cantidad de fechas inválidas: 391


In [47]:
display(
    df.loc[fechas_invalidas.index, ["Observation date"]].head(20)
)

,Observation date
279,13.03.2024
280,13.03.2024
281,13.03.2024
282,13.03.2024
283,13.03.2024
284,13.03.2024
285,13.03.2024
286,13.03.2024
289,14.03.2024
290,14.03.2024


In [63]:
mask = df_limpio["Observation date"].isna()

indices = df_limpio.index[mask]

fechas_originales = df.loc[indices, "Observation date"]

fechas_recuperadas = pd.to_datetime(
    fechas_originales,
    format="%d.%m.%Y",
    errors="coerce")


In [64]:
df_limpio.loc[indices, "Observation date"] = fechas_recuperadas

In [65]:
print("Fechas faltantes:", df_limpio["Observation date"].isna().sum())
print("Tipo de dato:", df_limpio["Observation date"].dtype)

Fechas faltantes: 32
Tipo de dato: datetime64[ns]


In [67]:
fechas_pendientes = df.loc[
    df_limpio.index[df_limpio["Observation date"].isna()],
    "Observation date"
]

print(fechas_pendientes.value_counts(dropna=False))

Observation date
5 May 2024    9
2 may 2024    6
1 May 2024    5
6 May 2024    4
7 May 2024    4
3 May 2024    2
4 May 2024    1
8 May 2024    1
Name: count, dtype: int64


In [68]:
# Identificar las fechas que todavía faltan
mask = df_limpio["Observation date"].isna()

# Recuperar los valores originales
indices = df_limpio.index[mask]

fechas_originales = df.loc[indices, "Observation date"]

# Convertir las fechas escritas en inglés
fechas_recuperadas = pd.to_datetime(
    fechas_originales,
    format="%d %B %Y",
    errors="coerce"
)

# Actualizar el DataFrame limpio
df_limpio.loc[indices, "Observation date"] = fechas_recuperadas

In [69]:
print("Fechas faltantes:", df_limpio["Observation date"].isna().sum())

print("Tipo de dato:", df_limpio["Observation date"].dtype)

Fechas faltantes: 0
Tipo de dato: datetime64[ns]


In [70]:
display(df_limpio.head())

print("Dimensiones finales:", df_limpio.shape)
print("Duplicados:", df_limpio.duplicated().sum())
print("Valores faltantes por columna:")
print(df_limpio.isnull().sum())

,Animal type,Country,Weight kg,Body Length cm,Gender,Latitude,Longitude,Observation date,Data compiled by
0,Unknown,Unknown,0.295,20.0,Not determined,NaN,NaN,2024-03-01,James Johnson
1,Unknown,Unknown,0.295,20.0,Not determined,NaN,NaN,2024-03-02,James Johnson
2,European bison,Poland,930.000,335.0,male,52.828845,23.820144,2024-01-03,Anne Anthony
3,European bison,Poland,909.000,311.0,not determined,52.830509,23.826849,2024-01-03,Anne Anthony
4,European bison,Poland,581.000,277.0,female,52.834109,23.807093,2024-01-03,Anne Anthony


Dimensiones finales: (844, 9)
Duplicados: 0
Valores faltantes por columna:
Animal type          0
Country              0
Weight kg            0
Body Length cm       0
Gender               0
Latitude            92
Longitude           92
Observation date     0
Data compiled by     0
dtype: int64


### Limpieza de datos

Durante el proceso de limpieza se realizaron las siguientes actividades:

1. Se creó una copia del conjunto de datos original para conservar la información inicial.
2. Se eliminaron los registros duplicados, pasando de 167 duplicados identificados a cero.
3. Se eliminaron las columnas `Animal code` y `Animal name`, debido a que presentaban una cantidad excesiva de valores faltantes.
4. Se estandarizaron las categorías de animales y países, corrigiendo errores ortográficos y unificando abreviaturas.
5. Los valores faltantes de las variables categóricas se identificaron como `Unknown` o `Not determined`.
6. Los valores negativos de peso y longitud se trataron como datos faltantes y se imputaron utilizando la mediana.
7. Se corrigieron los formatos de las fechas, recuperando registros escritos con diferentes estructuras.
8. Los valores de coordenadas faltantes o geográficamente inválidos se conservaron como valores nulos, evitando inventar información.

**Resultado:** el conjunto de datos pasó de 1.011 registros y 11 columnas a 844 registros y 9 columnas. Se eliminaron los duplicados y se logró completar la variable de fecha. Las variables de latitud y longitud conservaron 92 valores faltantes cada una.

**3. NumPy**

In [73]:
#Crear una variable con np.where()

df_limpio["Clasificacion peso"] = np.where(df_limpio["Weight kg"] >= 100,"Peso alto","Peso bajo")
print(df_limpio["Clasificacion peso"].value_counts())

Clasificacion peso
Peso bajo    784
Peso alto     60
Name: count, dtype: int64


In [74]:
#Calcular una medida estadística con NumPy
promedio_peso = np.mean(df_limpio["Weight kg"])

print("El peso promedio de los animales es:", promedio_peso, "kg")

El peso promedio de los animales es: 46.29282227488152 kg


In [75]:
#median
mediana_peso = np.median(df_limpio["Weight kg"])

print("La mediana del peso es:", mediana_peso, "kg")

La mediana del peso es: 0.346 kg


In [77]:
promedio_sin_nan = np.nanmean(df_limpio["Weight kg"])

print("Promedio sin valores faltantes:", promedio_sin_nan)

mediana_sin_nan = np.nanmedian(df_limpio["Weight kg"])

print("Mediana sin valores faltantes:", mediana_sin_nan)

desviacion = np.std(df_limpio["Weight kg"])

print("Desviación estándar:", desviacion)

Promedio sin valores faltantes: 46.29282227488152
Mediana sin valores faltantes: 0.346
Desviación estándar: 167.77411195819042


Los resultados muestran que el peso promedio de los animales es de 46,29 kg, mientras que la mediana es de 0,346 kg. Esta diferencia indica que existen algunos animales con pesos considerablemente elevados que aumentan el promedio.

Por otra parte, la desviación estándar de 167,7 kg evidencia que los pesos presentan una alta variabilidad entre las observaciones.

En conclusión, los datos muestran una gran diferencia de peso entre las especies analizadas. Por esta razón, la mediana representa mejor el peso central o típico de los registros que el promedio.

**4.GroupBy**



In [78]:
#observacion por tipo de animal
observaciones_animales = df_limpio.groupby("Animal type").size()

print(observaciones_animales)

Animal type
European bison     63
Unknown            19
hedgehog          278
lynx               71
red squirrel      413
dtype: int64


In [79]:
#peso promedio por tipo de animal
peso_promedio_animal = df_limpio.groupby("Animal type")["Weight kg"].mean()

print(peso_promedio_animal)

Animal type
European bison    588.396825
Unknown             0.291737
hedgehog            0.797482
lynx               23.267606
red squirrel        0.297576
Name: Weight kg, dtype: float64


In [80]:
#peso promedio x pais
peso_promedio_pais = df_limpio.groupby("Country")["Weight kg"].mean()

print(peso_promedio_pais)

Country
Australia           0.494000
Austria             1.944041
CC                634.000000
Czech Republic     54.513911
Germany             0.456090
Hungary             2.883093
Poland            129.996125
Slovakia           32.449397
Unknown             0.295000
Name: Weight kg, dtype: float64


In [81]:
#observaciones x pais
observaciones_pais = df_limpio.groupby("Country").size()

print(observaciones_pais.sort_values(ascending=False))

Country
Poland            192
Germany           178
Slovakia          151
Hungary           118
Czech Republic    112
Austria            74
Unknown            11
Australia           4
CC                  4
dtype: int64


**Punto 5. Cuatro conclusiones generales**

## Conclusiones generales de la actividad

**1. La calidad de los datos es fundamental para obtener resultados confiables.**

Desde mi punto de vista, uno de los aprendizajes más importantes de esta actividad fue comprender que antes de realizar cualquier análisis es necesario revisar y limpiar la información. Encontré datos duplicados, valores faltantes, errores en los nombres de los animales y diferencias en la escritura de los países. Considero que, si no hubiera realizado estas correcciones, los resultados podrían haber sido poco precisos o llevarme a interpretaciones equivocadas.

**2. Las diferencias entre el promedio y la mediana permiten entender mejor los datos.**

Al analizar los pesos de los animales, observé que el promedio fue de 46,29 kg, mientras que la mediana fue de 0,346 kg. Esta diferencia me llamó la atención porque demuestra que existen animales con pesos muy elevados que afectan considerablemente el promedio. Desde mi perspectiva, la mediana representa mejor el valor central de los registros, mientras que la desviación estándar de 167,7 kg evidencia una gran variabilidad entre los pesos.

**3. Las especies tienen características muy diferentes y su distribución no es uniforme.**

Al utilizar `groupby()`, identifiqué que la ardilla roja fue la especie con mayor número de observaciones, con 413 registros, seguida del erizo con 278. También encontré diferencias importantes en los pesos promedio: el bisonte europeo registró aproximadamente 588,40 kg, mientras que la ardilla roja presentó un promedio de 0,30 kg. Esto me permitió comprender la importancia de analizar los datos por categorías y no solamente de manera general.

**4. La distribución geográfica de los registros permite identificar patrones, pero también datos que requieren validación.**

Al analizar la información por países, encontré que Polonia tiene la mayor cantidad de observaciones, con 192 registros, seguida de Alemania con 178. También observé diferencias importantes en los pesos promedio entre países. Sin embargo, considero que es necesario tener precaución al interpretar estos resultados, porque algunas categorías como CC y Australia requieren verificación. Esto me permitió entender que no basta con obtener resultados estadísticos: también es importante cuestionar la información y evaluar si tiene sentido dentro del contexto analizado.

### Mi conclusión

A pesar que estoy desde cero en este mundo del analisis esta actividad me permitió comprender que el análisis de datos no consiste únicamente en aplicar funciones de Python, sino en saber interpretar la información, identificar inconsistencias y tomar decisiones sobre cómo trabajar los datos. También entendí que herramientas como Pandas y NumPy facilitan la organización y el análisis de grandes cantidades de información, pero que los resultados siempre deben analizarse teniendo en cuenta su contexto y calidad. Aunque no ha sido facil espero seguir interiorizando la información.